In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb
import joblib
import os

print("All imports successful")

All imports successful


In [2]:
def parse_orphanet_symptoms(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()
    
    records = []
    
    for disorder in root.iter("Disorder"):
        name_el = disorder.find("Name")
        if name_el is None:
            continue
        disease_name = name_el.text.strip()
        
        symptoms = []
        for sign in disorder.iter("HPODisorderAssociation"):
            hpo = sign.find(".//HPOTerm")
            if hpo is not None:
                symptoms.append(hpo.text.strip().lower())
        
        if len(symptoms) >= 3:  # only diseases with 3+ symptoms
            records.append({
                "disease": disease_name,
                "symptoms": symptoms
            })
    
    return records

filepath = "../datasets/en_product4.xml"
records = parse_orphanet_symptoms(filepath)
print(f"Total diseases parsed: {len(records)}")
print(f"Sample: {records[0]}")

Total diseases parsed: 4293
Sample: {'disease': 'Alexander disease', 'symptoms': ['macrocephaly', 'intellectual disability', 'seizure', 'spasticity', 'agenesis of corpus callosum', 'hyperreflexia', 'megalencephaly', 'failure to thrive', 'frontal bossing', 'nausea and vomiting', 'abnormality of speech or vocalization', 'clonus', 'eeg abnormality', 'sleep abnormality', 'scoliosis', 'abnormal pyramidal sign', 'large face', 'abnormality of eye movement', 'ptosis', 'nystagmus', 'diplopia', 'emotional lability', 'depression', 'hyperhidrosis', 'ataxia', 'dysarthria', 'gait disturbance', 'tremor', 'dysphonia', 'dysphagia', 'constipation', 'hypothermia', 'aphasia', 'tetraplegia', 'cerebral calcification', 'hypotension', 'kyphosis', 'sleep apnea', 'facial palsy', 'recurrent singultus', 'high palate', 'hydrocephalus', 'short neck', 'diabetes mellitus', 'hypothyroidism', 'hypertension', 'precocious puberty', 'osteopenia', 'hypotonia', 'muscle weakness', 'sudden cardiac death', 'chorea', 'respirato

In [3]:
# Collect all unique symptoms
all_symptoms = set()
for record in records:
    for symptom in record["symptoms"]:
        all_symptoms.add(symptom)

symptom_columns = sorted(list(all_symptoms))
print(f"Total unique symptoms found: {len(symptom_columns)}")
print(f"Sample symptoms: {symptom_columns[:10]}")

Total unique symptoms found: 8698
Sample symptoms: ['1-2 finger syndactyly', '1-2 toe complete cutaneous syndactyly', '1-2 toe syndactyly', '1-5 finger complete cutaneous syndactyly', '1-5 finger syndactyly', '1-minute apgar score of 0', '1-minute apgar score of 1', '11 pairs of ribs', '2-3 finger syndactyly', '2-3 toe cutaneous syndactyly']


In [4]:

rows = []
labels = []

for record in records:
    disease = record["disease"]
    symptoms = set(record["symptoms"])
    
    # Create binary row — 1 if symptom present, 0 if not
    row = [1 if col in symptoms else 0 for col in symptom_columns]
    rows.append(row)
    labels.append(disease)

X = np.array(rows)
y = np.array(labels)

print(f"Feature matrix shape: {X.shape}")
print(f"Number of disease classes: {len(set(y))}")


Feature matrix shape: (4293, 8698)
Number of disease classes: 4293


In [6]:

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Label encoding done")
print(f"Sample: {y[:3]} → {y_encoded[:3]}")

# ─────────────────────────────────────────────────────────
# Cell 6 — Train test split
# ─────────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42
)

print(f"Train size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")

Label encoding done
Sample: ['Alexander disease' 'Alpha-mannosidosis' 'Aspartylglucosaminuria'] → [288 311 410]
Train size: 3434
Test size: 859


In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")
xgb_model.fit(
    X, y_encoded,
    verbose=50
)
print("Training complete")

Training XGBoost...


c:\Users\tejak\hackathon\backend\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:07:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
# ─────────────────────────────────────────────────────────
# Cell 8 — Save the model (no test evaluation since 1 sample per class)
# ─────────────────────────────────────────────────────────

# Save the trained model
model_path = "../models_saved/xgb_disease_classifier.pkl"
joblib.dump(xgb_model, model_path)
print(f"Model saved to {model_path}")

# Save the label encoder
encoder_path = "../models_saved/label_encoder.pkl"
joblib.dump(label_encoder, encoder_path)
print(f"Label encoder saved to {encoder_path}")

# Save symptom columns for inference
symptoms_path = "../models_saved/symptom_columns.pkl"
joblib.dump(symptom_columns, symptoms_path)
print(f"Symptom columns saved to {symptoms_path}")

print("Model trained on all data. For evaluation, consider cross-validation or more data per class.")